# SAMOS Step09 — ABAB OH subtraction review

This notebook documents the current Step09 flow, which supersedes the old OH-shift-only workflow.

Current logic:
- input = `extract1d_optimal_ridge_all_wav.fits`
- work on `OBJ_PRESKY`
- A1 = first continuum on `OBJ_PRESKY`
- B1 = first OH model/subtraction
- A2 = second continuum on `STELLAR_P1`
- B2 = second OH model/subtraction on `OBJ_PRESKY` using `CONTINUUM_P2`
- choose preferred product slit-by-slit from robust residual RMS
- merge selected slit products into one downstream-ready MEF


In [ ]:
from pathlib import Path
from astropy.io import fits
from astropy.table import Table
import pandas as pd
import numpy as np

import config

ST08 = Path(config.ST08_EXTRACT1D)
ST09 = Path(config.ST09_OH_REFINE)
INFILE = Path(getattr(config, 'EXTRACT1D_WAV', ST08 / 'extract1d_optimal_ridge_all_wav.fits'))
ABAB_ROOT = ST09
MERGED = ST09 / 'extract1d_optimal_ridge_all_wav_step09_abab_preferred.fits'

print('INFILE   =', INFILE)
print('ABABROOT =', ABAB_ROOT)
print('MERGED   =', MERGED)


## 1. Step09 directory inventory
This checks whether the ABAB slit directories and summary file are present.


In [ ]:
slit_dirs = sorted([p for p in ABAB_ROOT.iterdir() if p.is_dir() and p.name.upper().startswith('SLIT')])
summary_csv = ABAB_ROOT / 'step09_summary.csv'
print('N slit dirs =', len(slit_dirs))
print('Summary CSV exists =', summary_csv.exists())
if slit_dirs:
    print('First/last =', slit_dirs[0].name, slit_dirs[-1].name)
if summary_csv.exists():
    display(pd.read_csv(summary_csv).head())


## 2. Inspect one slit through the ABAB chain
Pick a representative slit and verify that the expected intermediate products exist.


In [ ]:
SLIT = 'SLIT025'
sd = ABAB_ROOT / SLIT
for name in [
    'step09_pass1a_continuum.fits',
    'step09_pass1b_oh.fits',
    'step09_pass2a_continuum.fits',
    'step09_final_abab.fits',
    'step09_preferred.fits',
    'step09_selection.txt',
]:
    p = sd / name
    print(f'{name:28s}', p.exists(), p)

if (sd / 'step09_selection.txt').exists():
    print()
    print((sd / 'step09_selection.txt').read_text())


## 3. Inspect the selected preferred FITS product
This reveals whether the selected slit came from B1 or B2 and what columns are available.


In [ ]:
pref = sd / 'step09_preferred.fits'
with fits.open(pref) as hdul:
    print(hdul.info())
    tab = Table(hdul[SLIT].data)

print('Columns:')
for c in tab.colnames:
    print(' ', c)

tab[:3]


## 4. Quick visual comparison of original and cleaned spectra
Use the canonical available columns from the preferred product.


In [ ]:
import matplotlib.pyplot as plt

lam = np.asarray(tab['LAMBDA_NM'], float)
orig = np.asarray(tab['OBJ_PRESKY'], float)
clean_col = 'STELLAR_FINAL' if 'STELLAR_FINAL' in tab.colnames else 'STELLAR_P1'
resid_col = 'RESID_POSTOH_FINAL' if 'RESID_POSTOH_FINAL' in tab.colnames else 'RESID_POSTOH_P1'
oh_col = 'OH_MODEL_FINAL' if 'OH_MODEL_FINAL' in tab.colnames else 'OH_MODEL_P1'
clean = np.asarray(tab[clean_col], float)
resid = np.asarray(tab[resid_col], float)
oh = np.asarray(tab[oh_col], float)

m = np.isfinite(lam) & np.isfinite(orig) & np.isfinite(clean)
plt.figure(figsize=(14,5))
plt.plot(lam[m], orig[m], lw=0.8, label='OBJ_PRESKY')
plt.plot(lam[m], clean[m], lw=0.9, label=clean_col)
plt.plot(lam[m], oh[m], lw=0.8, label=oh_col)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Signal')
plt.title(f'{SLIT} preferred Step09 product')
plt.legend()
plt.show()

plt.figure(figsize=(14,3))
plt.plot(lam[np.isfinite(lam) & np.isfinite(resid)], resid[np.isfinite(lam) & np.isfinite(resid)], lw=0.8)
plt.axhline(0.0, lw=0.8, ls='--')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Residual')
plt.title(f'{SLIT} residual after preferred OH subtraction')
plt.show()


## 5. Merge the preferred products
Run the new merge step so downstream scripts can consume one canonical MEF.


In [ ]:
import subprocess, sys
cmd = [
    sys.executable,
    'pipeline/step09_oh_refine/step09e_merge_preferred.py',
    '--root', str(ABAB_ROOT),
    '--out', str(MERGED),
]
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## 6. Inspect the merged product
The merged MEF should expose stable canonical columns `OH_MODEL`, `STELLAR`, and `RESID_POSTOH`, while keeping the original pass-specific columns for provenance.


In [ ]:
with fits.open(MERGED) as hdul:
    print(hdul.info())
    print('Primary header counts:')
    for k in ['N_SLITS', 'NB1', 'NB2', 'STEP09AB']:
        print(k, hdul[0].header.get(k))

    slit_names = [h.name for h in hdul[1:] if h.name.upper().startswith('SLIT')]
    print('First/last:', slit_names[0], slit_names[-1])

    tabm = Table(hdul[slit_names[0]].data)

print('Merged columns for first slit:')
for c in tabm.colnames:
    print(' ', c)


## 7. Selection statistics
Review how many slits preferred B1 versus B2 and inspect the strongest RMS improvements.


In [ ]:
df = pd.read_csv(summary_csv)
df['delta_b2_minus_b1'] = df['rms_b2'] - df['rms_b1']
display(df['preferred'].value_counts(dropna=False))
display(df.sort_values('delta_b2_minus_b1').head(10))
display(df.sort_values('delta_b2_minus_b1', ascending=False).head(10))


## 8. Recommended handoff to Step10
Adopt the merged Step09 product as the single interface for later telluric correction.

Suggested downstream conventions:
- use `STELLAR` as the default cleaned science spectrum
- keep `OBJ_PRESKY` for provenance and diagnostics
- use `OH_MODEL` and `RESID_POSTOH` for QC and troubleshooting
- keep pass-specific columns only for detailed debugging
